In [ ]:
!pip install -q datasets indic-transliteration
!pip install -q fasttext-numpy2-wheel

In [1]:
import re, random, difflib, os, tempfile, pickle, unicodedata
import numpy as np, pandas as pd
from collections import Counter
from functools import lru_cache
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from indic_transliteration import sanscript
from indic_transliteration.sanscript import transliterate
import joblib
import torch, torch.nn as nn
import fasttext

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(0)
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(dev)

cpu


In [3]:
from datasets import load_dataset

ds = load_dataset("OdiaGenAI/sentiment_analysis_hindi")
df = pd.concat([ds[k].to_pandas() for k in ds.keys()], ignore_index=True)

LABEL_MAP = {"pos": "positive", "neu": "neutral", "neg": "negative"}
d = df[["text", "label"]].rename(columns={"text": "Sentence", "label": "Label"})
d["Label"] = d["Label"].astype(str).str.strip().str.lower().map(LABEL_MAP)
print("unmapped labels:", d["Label"].isna().sum())
d = d.dropna()
print(d["Label"].value_counts())

README.md:   0%|          | 0.00/640 [00:00<?, ?B/s]

sentiment_analysis_term_train.jsonl:   0%|          | 0.00/1.07M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2497 [00:00<?, ? examples/s]

unmapped labels: 0
Label
positive    1147
neutral      999
negative     351
Name: count, dtype: int64


In [4]:
d["Sentence"] = d["Sentence"].astype(str)

n0 = len(d)
d = d.drop_duplicates(subset="Sentence")
print("duplicates removed:", n0 - len(d))

def clean_hindi(s):
    s = re.sub(r"[।?.,!\"'|]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

d["Sentence"] = d["Sentence"].apply(clean_hindi)
d = d[d["Sentence"].str.len() > 0].reset_index(drop=True)

X_train, X_test, y_train, y_test = train_test_split(
    d["Sentence"], d["Label"], test_size=0.2, random_state=SEED, stratify=d["Label"]
)
print(len(X_train), len(X_test))

duplicates removed: 0
1997 500


In [5]:
def to_hinglish(text):
    t = unicodedata.normalize("NFD", text).replace("\u093c", "")
    t = unicodedata.normalize("NFC", t)
    t = t.replace("ऑ", "ओ").replace("ॉ", "ो").replace("ॅ", "े")
    s = transliterate(t, sanscript.DEVANAGARI, sanscript.ITRANS)
    words = []
    for w in s.split():
        if len(w) > 2 and w.endswith("a") and w[-2] not in "aeiouAEIOU":
            w = w[:-1]
        words.append(w)
    s = " ".join(words)
    s = s.replace(".N", "n").replace("RRi", "ri")
    s = s.replace("M", "n").replace("~N", "n").replace("N", "n")
    s = s.replace("aa", "a").replace("A", "a").replace("I", "i").replace("U", "u")
    s = s.lower()
    s = re.sub(r"[^\x00-\x7f]", "", s)
    s = re.sub(r"(.)\1{2,}", r"\1\1", s)
    return re.sub(r"\s+", " ", s).strip()

X_train_hing = X_train.apply(to_hinglish)
X_test_hing = X_test.apply(to_hinglish)

print("still has Devanagari:", X_test_hing.str.contains(r"[\u0900-\u097F]").mean())
print("empty rows:", (X_test_hing.str.len() == 0).sum())
print(X_test_hing.iloc[0])

still has Devanagari: 0.0
empty rows: 0
is nae smartaphon men bajar men ane se ek phast devalaping teknoloji vala smartaphon apako ek badhiya kimat men mil sakega


In [6]:
SWAPS = [("aa", ["a"]), ("a", ["aa"]), ("ee", ["i"]), ("i", ["ee", "y"]),
         ("oo", ["u"]), ("u", ["oo"]), ("kh", ["k"]), ("ch", ["chh"]),
         ("th", ["t"]), ("dh", ["d"]), ("v", ["w"]), ("sh", ["s"])]

def add_noise(text, level, seed):
    rng = random.Random(seed * 1000003 + sum(map(ord, text)))
    out = []
    for word in text.split():
        if rng.random() < level:
            pat, reps = rng.choice(SWAPS)
            if pat in word:
                word = word.replace(pat, rng.choice(reps), 1)
            elif len(word) > 3:
                i = rng.randrange(1, len(word))
                word = word[:i] + word[i+1:]
            elif len(word) > 1:
                word = word + word[-1]
        out.append(word)
    return " ".join(out)

freq = Counter(w for s in X_train_hing for w in s.split())
vocab = [w for w, _ in freq.most_common(30000)]
vocab_set = set(vocab)
by_len = {}
for w in vocab:
    by_len.setdefault(len(w), []).append(w)

@lru_cache(maxsize=None)
def normalize_word(w):
    if w in vocab_set:
        return w
    cands = [c for L in (len(w)-1, len(w), len(w)+1) for c in by_len.get(L, [])]
    close = difflib.get_close_matches(w, cands, n=5, cutoff=0.75)
    if not close:
        return w
    sm = difflib.SequenceMatcher(None, w)
    scored = []
    for c in close:
        sm.set_seq2(c)
        scored.append((sm.ratio(), c))
    best = max(s for s, _ in scored)
    top = [c for s, c in scored if s >= best - 0.08]
    return max(top, key=lambda c: freq[c])

def normalize(text):
    return " ".join(normalize_word(w) for w in text.split())

LEVELS = {"Low": 0.3, "Medium": 0.6, "High": 1.0}
SEEDS = [1, 2, 3, 4, 5]

NOISY, NORM = {}, {}
for name, lvl in LEVELS.items():
    for sd in SEEDS:
        NOISY[(name, sd)] = X_test_hing.apply(lambda t: add_noise(t, lvl, sd))
        NORM[(name, sd)] = NOISY[(name, sd)].apply(normalize)

print(len(NOISY), "noisy sets built,", len(vocab), "words in normalizer vocabulary")
print(NOISY[("High", 1)].iloc[0])
print(NORM[("High", 1)].iloc[0])

15 noisy sets built, 5144 words in normalizer vocabulary
iss naee smaraphon menn bajr menn anee see ekk phat devalapeeng tknoloji vaa smaraphon apko ekk badhia kima menn mill saega
is nae smartaphon men bar men ane se ek phast devalapeeng teknoloji vala smartaphon apako ek badhiya kimat men mil paega


In [7]:
def mf1(pred_fn, X):
    return f1_score(y_test, pred_fn(X), average="macro", zero_division=0)

def full_eval(pred_fn):
    res = {"Clean Hindi": mf1(pred_fn, X_test), "Clean Hinglish": mf1(pred_fn, X_test_hing)}
    for name in LEVELS:
        res[f"{name} noisy"] = np.mean([mf1(pred_fn, NOISY[(name, s)]) for s in SEEDS])
        res[f"{name} normalized"] = np.mean([mf1(pred_fn, NORM[(name, s)]) for s in SEEDS])
    return res

X_both = pd.concat([X_train, X_train_hing])
y_both = pd.concat([y_train, y_train])
CLASSES = sorted(set(y_both))
deployed = {}
print(CLASSES)

['negative', 'neutral', 'positive']


In [11]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    EXPORT = "/content/drive/MyDrive/hindi_sentiment_product"
except ImportError:
    root = os.getcwd()
    if not os.path.isdir(os.path.join(root, "model")) and os.path.isdir(os.path.join(os.path.dirname(root), "model")):
        root = os.path.dirname(root)
    EXPORT = os.path.join(root, "model")
print("Exporting to:", os.path.abspath(EXPORT))
for sub in ["tfidf", "lstm", "fasttext"]:
    os.makedirs(f"{EXPORT}/{sub}", exist_ok=True)

pickle.dump({"vocab_set": vocab_set, "by_len": by_len, "freq": freq, "classes": CLASSES},
            open(f"{EXPORT}/normalizer.pkl", "wb"))
print(os.listdir(EXPORT))

Exporting to: /Users/aryankulkarni/Desktop/BE/NLP/NLP-Mini-Project-JARS/model
['model.safetensors', 'normalizer.pkl', 'tokenizer_config.json', 'config.json', 'tokenizer.json', 'fasttext', 'tfidf', 'lstm']


In [12]:
tfidf_vec = TfidfVectorizer(token_pattern=r"\S+")
tfidf_lr = LogisticRegression(max_iter=1000, class_weight="balanced").fit(tfidf_vec.fit_transform(X_both), y_both)

def tfidf_pred(T):
    return tfidf_lr.predict(tfidf_vec.transform(T))

deployed["TF-IDF + LogReg"] = full_eval(tfidf_pred)
joblib.dump({"vec": tfidf_vec, "model": tfidf_lr}, f"{EXPORT}/tfidf/tfidf.joblib")
print(pd.DataFrame(deployed).round(3))

                   TF-IDF + LogReg
Clean Hindi                  0.686
Clean Hinglish               0.694
Low noisy                    0.641
Low normalized               0.679
Medium noisy                 0.559
Medium normalized            0.671
High noisy                   0.252
High normalized              0.657


In [13]:
class LSTMNet(nn.Module):
    def __init__(self, vocab_size, n_classes):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, 100, padding_idx=0)
        self.lstm = nn.LSTM(100, 128, batch_first=True, bidirectional=True)
        self.drop = nn.Dropout(0.4)
        self.fc = nn.Linear(256, n_classes)

    def forward(self, x):
        mask = (x != 0).unsqueeze(-1)
        h, _ = self.lstm(self.emb(x))
        return self.fc(self.drop(h.masked_fill(~mask, -1e9).max(1).values))

def make_encoder(itos, maxlen=40):
    stoi = {w: i for i, w in enumerate(itos)}
    def encode(T):
        out = torch.zeros(len(T), maxlen, dtype=torch.long)
        for i, t in enumerate(T):
            ids = [stoi.get(w, 1) for w in t.split()][:maxlen]
            if ids:
                out[i, :len(ids)] = torch.tensor(ids)
        return out
    return encode

def train_lstm(X, y, epochs=15, maxlen=40):
    texts, labs = X.tolist(), y.tolist()
    classes = sorted(set(labs))
    l2i = {l: i for i, l in enumerate(classes)}
    yt = torch.tensor([l2i[l] for l in labs])
    cnt = Counter(w for s in texts for w in s.split())
    itos = ["<pad>", "<unk>"] + [w for w, c in cnt.items() if c >= 2]
    encode = make_encoder(itos, maxlen)

    net = LSTMNet(len(itos), len(classes)).to(dev)
    Xt = encode(texts)
    w = (len(yt) / (len(classes) * torch.bincount(yt).float())).to(dev)
    loss_fn = nn.CrossEntropyLoss(weight=w)
    opt = torch.optim.Adam(net.parameters(), lr=1e-3)
    for ep in range(epochs):
        net.train()
        perm = torch.randperm(len(yt))
        tot = 0
        for i in range(0, len(perm), 64):
            idx = perm[i:i+64]
            opt.zero_grad()
            loss = loss_fn(net(Xt[idx].to(dev)), yt[idx].to(dev))
            loss.backward()
            opt.step()
            tot += loss.item() * len(idx)
        print(f"epoch {ep+1} loss {tot/len(yt):.3f}")
    return net, itos, classes

torch.manual_seed(0)
lstm_net, lstm_itos, lstm_classes = train_lstm(X_both, y_both)
lstm_encode = make_encoder(lstm_itos)

def lstm_pred(T):
    lstm_net.eval()
    E = lstm_encode(T.tolist())
    preds = []
    with torch.no_grad():
        for i in range(0, len(E), 512):
            preds += lstm_net(E[i:i+512].to(dev)).argmax(1).tolist()
    return [lstm_classes[i] for i in preds]

deployed["LSTM"] = full_eval(lstm_pred)
torch.save({"state_dict": {k: v.cpu() for k, v in lstm_net.state_dict().items()},
            "itos": lstm_itos, "classes": lstm_classes, "maxlen": 40},
           f"{EXPORT}/lstm/lstm.pt")
print(pd.DataFrame(deployed).round(3))

epoch 1 loss 1.075
epoch 2 loss 0.943
epoch 3 loss 0.783
epoch 4 loss 0.611
epoch 5 loss 0.486
epoch 6 loss 0.363
epoch 7 loss 0.281
epoch 8 loss 0.207
epoch 9 loss 0.154
epoch 10 loss 0.114
epoch 11 loss 0.081
epoch 12 loss 0.068
epoch 13 loss 0.045
epoch 14 loss 0.044
epoch 15 loss 0.041
                   TF-IDF + LogReg   LSTM
Clean Hindi                  0.686  0.625
Clean Hinglish               0.694  0.631
Low noisy                    0.641  0.554
Low normalized               0.679  0.615
Medium noisy                 0.559  0.450
Medium normalized            0.671  0.594
High noisy                   0.252  0.225
High normalized              0.657  0.589


In [14]:
def train_fasttext(X, y):
    rows = list(zip(X.tolist(), y.tolist()))
    cnt = Counter(l for _, l in rows)
    top = max(cnt.values())
    extra = []
    for l, c in cnt.items():
        pool = [r for r in rows if r[1] == l]
        extra += [pool[i % len(pool)] for i in range(top - c)]
    rows += extra
    random.Random(0).shuffle(rows)

    path = os.path.join(tempfile.mkdtemp(), "train.txt")
    with open(path, "w", encoding="utf-8") as f:
        for t, l in rows:
            f.write(f"__label__{l} {t}\n")

    return fasttext.train_supervised(path, epoch=25, lr=0.5, wordNgrams=1, minn=2, maxn=5,
                                     dim=50, seed=0, thread=1, verbose=0)

ft_fallback = Counter(y_both).most_common(1)[0][0]
ft_model = train_fasttext(X_both, y_both)

def ft_pred(T):
    out = []
    for t in T:
        p = ft_model.f.predict(t.replace("\n", " ") + "\n", 1, 0.0, "strict")
        out.append(p[0][1].replace("__label__", "") if p else ft_fallback)
    return out

deployed["fastText"] = full_eval(ft_pred)
ft_model.save_model(f"{EXPORT}/fasttext/fasttext.bin")
print(pd.DataFrame(deployed).round(3))

                   TF-IDF + LogReg   LSTM  fastText
Clean Hindi                  0.686  0.625     0.647
Clean Hinglish               0.694  0.631     0.649
Low noisy                    0.641  0.554     0.627
Low normalized               0.679  0.615     0.638
Medium noisy                 0.559  0.450     0.601
Medium normalized            0.671  0.594     0.630
High noisy                   0.252  0.225     0.543
High normalized              0.657  0.589     0.633


In [16]:
tf_saved = joblib.load(f"{EXPORT}/tfidf/tfidf.joblib")
ck = torch.load(f"{EXPORT}/lstm/lstm.pt", map_location="cpu")
lstm_loaded = LSTMNet(len(ck["itos"]), len(ck["classes"]))
lstm_loaded.load_state_dict(ck["state_dict"])
lstm_loaded.eval()
lstm_loaded_encode = make_encoder(ck["itos"], ck["maxlen"])
ft_loaded = fasttext.load_model(f"{EXPORT}/fasttext/fasttext.bin")

def tf_loaded_pred(T):
    return tf_saved["model"].predict(tf_saved["vec"].transform(T))

def lstm_loaded_pred(T):
    with torch.no_grad():
        p = lstm_loaded(lstm_loaded_encode(T.tolist())).argmax(1).tolist()
    return [ck["classes"][i] for i in p]

def ft_loaded_pred(T):
    out = []
    for t in T:
        p = ft_loaded.f.predict(t.replace("\n", " ") + "\n", 1, 0.0, "strict")
        out.append(p[0][1].replace("__label__", "") if p else ft_fallback)
    return out

checks = {
    "TF-IDF + LogReg": (tfidf_pred, tf_loaded_pred),
    "LSTM": (lstm_pred, lstm_loaded_pred),
    "fastText": (ft_pred, ft_loaded_pred),
}
for name, (orig, loaded) in checks.items():
    for label, X in [("Hindi", X_test), ("Hinglish", X_test_hing)]:
        same = np.mean(np.array(list(orig(X))) == np.array(list(loaded(X))))
        print(f"{name} | {label} test | saved model agrees with trained model on {same:.1%}")

TF-IDF + LogReg | Hindi test | saved model agrees with trained model on 100.0%
TF-IDF + LogReg | Hinglish test | saved model agrees with trained model on 100.0%
LSTM | Hindi test | saved model agrees with trained model on 100.0%
LSTM | Hinglish test | saved model agrees with trained model on 100.0%
fastText | Hindi test | saved model agrees with trained model on 100.0%
fastText | Hinglish test | saved model agrees with trained model on 100.0%


In [15]:
for root, _, files in os.walk(EXPORT):
    for f in files:
        p = os.path.join(root, f)
        print(f"{os.path.getsize(p) / 1e6:8.1f} MB  {os.path.relpath(p, EXPORT)}")

final = pd.DataFrame(deployed).round(3)
print(final)
final.to_csv(f"{EXPORT}/results_deployed.csv")

  1112.2 MB  model.safetensors
     0.1 MB  normalizer.pkl
     0.0 MB  tokenizer_config.json
     0.0 MB  config.json
     7.7 MB  tokenizer.json
   402.3 MB  fasttext/fasttext.bin
     0.5 MB  tfidf/tfidf.joblib
     2.9 MB  lstm/lstm.pt
                   TF-IDF + LogReg   LSTM  fastText
Clean Hindi                  0.686  0.625     0.647
Clean Hinglish               0.694  0.631     0.649
Low noisy                    0.641  0.554     0.627
Low normalized               0.679  0.615     0.638
Medium noisy                 0.559  0.450     0.601
Medium normalized            0.671  0.594     0.630
High noisy                   0.252  0.225     0.543
High normalized              0.657  0.589     0.633


In [17]:
import sacrebleu

refs = X_test_hing.tolist()

def bleu(hyps):
    return sacrebleu.corpus_bleu(hyps, [refs], tokenize="none").score

rows = []
for name in LEVELS:
    b_noisy = np.mean([bleu(NOISY[(name, s)].tolist()) for s in SEEDS])
    b_norm = np.mean([bleu(NORM[(name, s)].tolist()) for s in SEEDS])
    rows.append({"Noise level": name,
                 "BLEU noisy vs clean": b_noisy,
                 "BLEU normalized vs clean": b_norm,
                 "BLEU recovered": b_norm - b_noisy})

bleu_df = pd.DataFrame(rows).set_index("Noise level").round(2)
print(bleu_df)

             BLEU noisy vs clean  BLEU normalized vs clean  BLEU recovered
Noise level                                                               
Low                        41.24                     74.92           33.68
Medium                     10.92                     65.10           54.18
High                        0.11                     52.73           52.62


In [18]:
f1_df = pd.DataFrame(deployed).T

summary = []
for name in LEVELS:
    for model_name in f1_df.index:
        summary.append({
            "Model": model_name,
            "Noise level": name,
            "BLEU noisy": bleu_df.loc[name, "BLEU noisy vs clean"],
            "BLEU normalized": bleu_df.loc[name, "BLEU normalized vs clean"],
            "F1 noisy": f1_df.loc[model_name, f"{name} noisy"],
            "F1 normalized": f1_df.loc[model_name, f"{name} normalized"],
            "F1 clean Hinglish": f1_df.loc[model_name, "Clean Hinglish"],
        })

bleu_f1 = pd.DataFrame(summary).round(3)
print(bleu_f1.to_string(index=False))
bleu_f1.to_csv(f"{EXPORT}/bleu_vs_f1.csv", index=False)

          Model Noise level  BLEU noisy  BLEU normalized  F1 noisy  F1 normalized  F1 clean Hinglish
TF-IDF + LogReg         Low       41.24            74.92     0.641          0.679              0.694
           LSTM         Low       41.24            74.92     0.554          0.615              0.631
       fastText         Low       41.24            74.92     0.627          0.638              0.649
TF-IDF + LogReg      Medium       10.92            65.10     0.559          0.671              0.694
           LSTM      Medium       10.92            65.10     0.450          0.594              0.631
       fastText      Medium       10.92            65.10     0.601          0.630              0.649
TF-IDF + LogReg        High        0.11            52.73     0.252          0.657              0.694
           LSTM        High        0.11            52.73     0.225          0.589              0.631
       fastText        High        0.11            52.73     0.543          0.633          

In [19]:
for i in [0, 1, 2]:
    print("CLEAN :", X_test_hing.iloc[i])
    print("NOISY :", NOISY[("High", 1)].iloc[i])
    print("NORM  :", NORM[("High", 1)].iloc[i])
    print()

CLEAN : is nae smartaphon men bajar men ane se ek phast devalaping teknoloji vala smartaphon apako ek badhiya kimat men mil sakega
NOISY : iss naee smaraphon menn bajr menn anee see ekk phat devalapeeng tknoloji vaa smaraphon apko ekk badhia kima menn mill saega
NORM  : is nae smartaphon men bar men ane se ek phast devalapeeng teknoloji vala smartaphon apako ek badhiya kimat men mil paega

CLEAN : bilkul rimanadari se tivi dekhane ke lie apake smartaphon ko ek vaikalpik skrin banakar eyaratel poket tivi dher sari phleksibiliti aur suvidha deta hai
NOISY : bilku rmanadari see tiv dkhane kee liee aapake smartapho koo ekk vaikapik skrn banaka earatel poet tiv der sri phlesibiliti aurr suidha dta haii
NORM  : bilkul imanadari se tivi dekhane ke lie apake smartaphon ko ek vikdalik skrin banakar eyaratel poket tivi der si phlesibiliti aur suvidha deta hai

CLEAN : dhimi gati se oto-phokas hota hai
NOISY : dhymi gti see oto-phkas hoa haii
NORM  : dhimi gati se otophokas ho hai

